# 🚗 RoadSense — YOLOv11 Road Anomaly Detection (v2)

Retraining from v1 `best.pt` with **7 classes** (4 original + 3 new).

**Classes:** Pothole · Fallen-Barrier · Fallen-Cone · Fallen-Pole · Damaged-Traffic-Sign · Damaged-Street-Light · Faded-Road-Marking

**Strategy:** Download RoadFix (existing) + 2 Roboflow datasets + Mendeley markings from Drive → merge into one dataset → retrain from `best.pt`

> ⚠️ Before running: go to **Runtime → Change runtime type → T4 GPU**

## ✅ Step 1 — Check GPU

Confirms a GPU is assigned. If output shows 'No devices found', enable GPU in Runtime settings.

In [1]:
!nvidia-smi

Mon Jun 22 07:43:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Step 2 — Install Dependencies

Installs Ultralytics (YOLOv11) and Roboflow SDK.

In [2]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()


Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)


## 💾 Step 3 — Mount Google Drive

All checkpoints and the final model are saved directly to your Drive. This protects your progress if Colab disconnects.

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/RoadSense/runs'
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will save to: {DRIVE_PROJECT_PATH}')


Mounted at /content/drive
✅ Drive mounted. Checkpoints will save to: /content/drive/MyDrive/RoadSense/runs


In [7]:
import shutil
shutil.rmtree('/content/Street_light--1',      ignore_errors=True)
shutil.rmtree('/content/roadsense_merged',      ignore_errors=True)
# Clear new dataset dirs so re-runs don't get stale data
shutil.rmtree('/content/garbage-can-overflow',  ignore_errors=True)
shutil.rmtree('/content/garbage-clixe',         ignore_errors=True)
shutil.rmtree('/content/trash-bin-status-0ju0u',ignore_errors=True)
shutil.rmtree('/content/garbage-bin-gfxbz',     ignore_errors=True)
shutil.rmtree('/content/garbage-litter-detector',ignore_errors=True)
shutil.rmtree('/content/litter-street-images',  ignore_errors=True)

## ⚙️ Step 4 — Configuration

**Fill in your Roboflow private API key.** Everything else is pre-filled for the RoadFix dataset.

Find your private API key at: Roboflow → Settings → API Keys

In [8]:
# ── Original RoadFix dataset ─────────────────────────────────
RF_API_KEY          = 'DWcrraTLxI41cZAiSGTO'  # ← your private key
RF_WORKSPACE        = 'dequillaprojects'
RF_PROJECT          = 'roadfix'
RF_VERSION          = 3

# ── Existing sign / light datasets ───────────────────────────
RF_SIGNS_WORKSPACE  = 'matyworkspace'
RF_SIGNS_PROJECT    = 'damaged-traffic-signs'
RF_SIGNS_VERSION    = 2

RF_LIGHTS_WORKSPACE = 'godspeed-yqpeo'
RF_LIGHTS_PROJECT   = 'damaged-lights'
RF_LIGHTS_VERSION   = 1

# ── NEW: Litter on road ───────────────────────────────────────
RF_LITTER1_WORKSPACE = 'garbage-classification-yyarx'
RF_LITTER1_PROJECT   = 'garbage-litter-detector'
RF_LITTER1_VERSION   = 4

RF_LITTER2_WORKSPACE = 'kabml-images'
RF_LITTER2_PROJECT   = 'litter-street-images'
RF_LITTER2_VERSION   = 10

# ── NEW: Bin fullness ─────────────────────────────────────────
RF_BIN1_WORKSPACE   = 'mariswary-deepak-4ajr0'
RF_BIN1_PROJECT     = 'garbage-can-overflow'
RF_BIN1_VERSION     = 4

RF_BIN2_WORKSPACE   = 'dsoelma'
RF_BIN2_PROJECT     = 'garbage-clixe'
RF_BIN2_VERSION     = 6

RF_BIN3_WORKSPACE   = 'jamshid-salimov-s-workspace'
RF_BIN3_PROJECT     = 'trash-bin-status-0ju0u'
RF_BIN3_VERSION     = 5

RF_BIN4_WORKSPACE   = 'quality-control-defect-detection'
RF_BIN4_PROJECT     = 'garbage-bin-gfxbz'
RF_BIN4_VERSION     = 4

# ── v2 nano weights (transfer learning) ──────────────────────
V1_BEST_PT     = '/content/drive/MyDrive/RoadSense/runs/roadsense_nano/weights/best.pt'
USE_V1_WEIGHTS = True

# ── Mendeley faded markings ───────────────────────────────────
MENDELEY_DRIVE_PATH = '/content/drive/MyDrive/RoadSense/mendeley_markings/Attain/Attain'

# ── Training config ───────────────────────────────────────────
RUN_NAME           = 'roadsense_v3_9class'
EPOCHS             = 100
BATCH              = 32
IMG_SIZE           = 640
SAVE_EVERY         = 10
PATIENCE           = 10
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/RoadSense/runs'

import os
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print('Config set — 9 classes: Pothole, Fallen_Barrier, Fallen_Cone, Fallen_Pole,'
      ' Damaged_Traffic_Sign, Damaged_Street_Light, Faded_Road_Marking, Litter, Bin_Full')

Config set — 9 classes: Pothole, Fallen_Barrier, Fallen_Cone, Fallen_Pole, Damaged_Traffic_Sign, Damaged_Street_Light, Faded_Road_Marking, Litter, Bin_Full


In [6]:
import shutil
shutil.rmtree('/content/Street_light--1', ignore_errors=True)
shutil.rmtree('/content/roadsense_merged', ignore_errors=True)  # also clear merged

## 📥 Step 5 — Download All Datasets

Downloads RoadFix (4 original classes) + Damaged Traffic Signs + Street Lights from Roboflow, then copies the Mendeley faded markings from your Drive.

> For street lights we keep **only the `Nonworking` class** (renamed to `Damaged_Street_Light`). Working/Flicker labels are silently dropped during the merge.

In [11]:
from roboflow import Roboflow
import os, shutil, glob, yaml, random, math

rf = Roboflow(api_key=RF_API_KEY)

# ── 1. Download all datasets ──────────────────────────────────
ds_roadfix  = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov11')
ds_signs    = rf.workspace(RF_SIGNS_WORKSPACE).project(RF_SIGNS_PROJECT).version(RF_SIGNS_VERSION).download('yolov11')
ds_lights   = rf.workspace(RF_LIGHTS_WORKSPACE).project(RF_LIGHTS_PROJECT).version(RF_LIGHTS_VERSION).download('yolov11')
ds_litter1  = rf.workspace(RF_LITTER1_WORKSPACE).project(RF_LITTER1_PROJECT).version(RF_LITTER1_VERSION).download('yolov11')
ds_litter2  = rf.workspace(RF_LITTER2_WORKSPACE).project(RF_LITTER2_PROJECT).version(RF_LITTER2_VERSION).download('yolov11')
ds_bin1     = rf.workspace(RF_BIN1_WORKSPACE).project(RF_BIN1_PROJECT).version(RF_BIN1_VERSION).download('yolov11')
ds_bin2     = rf.workspace(RF_BIN2_WORKSPACE).project(RF_BIN2_PROJECT).version(RF_BIN2_VERSION).download('yolov11')
ds_bin3     = rf.workspace(RF_BIN3_WORKSPACE).project(RF_BIN3_PROJECT).version(RF_BIN3_VERSION).download('yolov11')
ds_bin4     = rf.workspace(RF_BIN4_WORKSPACE).project(RF_BIN4_PROJECT).version(RF_BIN4_VERSION).download('yolov11')

# ── 2. Read class names ────────────────────────────────────────
def read_classes(location):
    with open(os.path.join(location, 'data.yaml')) as f:
        return yaml.safe_load(f)['names']

classes_roadfix  = read_classes(ds_roadfix.location)
classes_signs    = read_classes(ds_signs.location)
classes_lights   = ['Not Working', 'Working']  # hardcoded — this dataset has no yaml names
classes_litter1  = read_classes(ds_litter1.location)
classes_litter2  = read_classes(ds_litter2.location)
classes_bin1     = read_classes(ds_bin1.location)
classes_bin2     = read_classes(ds_bin2.location)
classes_bin3     = read_classes(ds_bin3.location)
classes_bin4     = read_classes(ds_bin4.location)

print('RoadFix classes: ',  classes_roadfix)
print('Signs classes:   ',  classes_signs)
print('Litter1 classes: ',  classes_litter1)
print('Litter2 classes: ',  classes_litter2)
print('Bin1 classes:    ',  classes_bin1)
print('Bin2 classes:    ',  classes_bin2)
print('Bin3 classes:    ',  classes_bin3)
print('Bin4 classes:    ',  classes_bin4)

# ── 3. Global class list (9 classes) ──────────────────────────
GLOBAL_CLASSES = [
    'Pothole',               # 0
    'Fallen-Barrier',        # 1
    'Fallen-Cone',           # 2
    'Fallen-Pole',           # 3
    'Damaged_Traffic_Sign',  # 4
    'Damaged_Street_Light',  # 5
    'Faded_Road_Marking',    # 6
    'Litter',                # 7  ← NEW
    'Bin_Full',              # 8  ← NEW
]
print('\nGlobal classes:', GLOBAL_CLASSES)

# ── 4. Build class maps ────────────────────────────────────────
roadfix_map = {
    i: GLOBAL_CLASSES.index(name)
    for i, name in enumerate(classes_roadfix)
    if name in GLOBAL_CLASSES
}
signs_map   = {i: 4 for i in range(len(classes_signs))}  # all → Damaged_Traffic_Sign

lights_map  = {}
for i, name in enumerate(classes_lights):
    n = name.lower()
    if 'not working' in n or 'nonworking' in n or 'non_working' in n:
        lights_map[i] = 5
    else:
        lights_map[i] = None  # drop Working / Flicker

# Litter datasets — map ALL classes to Litter (7)
# These datasets only contain road litter so no filtering needed
litter1_map = {i: 7 for i in range(len(classes_litter1))}
litter2_map = {i: 7 for i in range(len(classes_litter2))}

# Bin datasets — keyword match → Bin_Full (8), drop everything else (empty, normal, half, etc.)
# ⚠️  Check the printed class names above and add any missing keywords here if a bin class
#     that should map to Bin_Full is being dropped.
FULL_KEYWORDS = ['full', 'overflow', 'overflowing', 'overfull', 'filled', 'over_flow']

def make_bin_map(class_list):
    m = {}
    for i, name in enumerate(class_list):
        n = name.lower().replace('-', '_').replace(' ', '_')
        m[i] = 8 if any(kw in n for kw in FULL_KEYWORDS) else None
    return m

bin1_map = make_bin_map(classes_bin1)
bin2_map = make_bin_map(classes_bin2)
bin3_map = make_bin_map(classes_bin3)
bin4_map = make_bin_map(classes_bin4)

print('\nLights map:', lights_map, '(None = dropped)')
print('Bin1 map:  ', bin1_map,   '(None = dropped)')
print('Bin2 map:  ', bin2_map,   '(None = dropped)')
print('Bin3 map:  ', bin3_map,   '(None = dropped)')
print('Bin4 map:  ', bin4_map,   '(None = dropped)')

# ── 5. Remap label files ───────────────────────────────────────
def remap_labels(src_location, class_map, out_base, split):
    src_img_dir = os.path.join(src_location, split, 'images')
    src_lbl_dir = os.path.join(src_location, split, 'labels')
    if not os.path.exists(src_img_dir):
        print(f'  Skipping {split} (not found): {src_img_dir}')
        return 0
    out_img_dir = os.path.join(out_base, split, 'images')
    out_lbl_dir = os.path.join(out_base, split, 'labels')
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    copied = 0
    for img_path in glob.glob(os.path.join(src_img_dir, '*')):
        fname   = os.path.splitext(os.path.basename(img_path))[0]
        lbl_src = os.path.join(src_lbl_dir, fname + '.txt')
        if not os.path.exists(lbl_src):
            continue
        new_lines = []
        with open(lbl_src) as lf:
            for line in lf:
                parts = line.strip().split()
                if not parts: continue
                old_id = int(parts[0])
                new_id = class_map.get(old_id)
                if new_id is None: continue
                new_lines.append(str(new_id) + ' ' + ' '.join(parts[1:]))
        if not new_lines:
            continue
        shutil.copy(img_path, out_img_dir)
        with open(os.path.join(out_lbl_dir, fname + '.txt'), 'w') as lf:
            lf.write('\n'.join(new_lines))
        copied += 1
    return copied

MERGED_DIR = '/content/roadsense_merged'

for split in ['train', 'valid', 'test']:
    n_rf = remap_labels(ds_roadfix.location,  roadfix_map,  MERGED_DIR, split)
    n_sg = remap_labels(ds_signs.location,    signs_map,    MERGED_DIR, split)
    n_lt = remap_labels(ds_lights.location,   lights_map,   MERGED_DIR, split)
    n_l1 = remap_labels(ds_litter1.location,  litter1_map,  MERGED_DIR, split)
    n_l2 = remap_labels(ds_litter2.location,  litter2_map,  MERGED_DIR, split)
    n_b1 = remap_labels(ds_bin1.location,     bin1_map,     MERGED_DIR, split)
    n_b2 = remap_labels(ds_bin2.location,     bin2_map,     MERGED_DIR, split)
    n_b3 = remap_labels(ds_bin3.location,     bin3_map,     MERGED_DIR, split)
    n_b4 = remap_labels(ds_bin4.location,     bin4_map,     MERGED_DIR, split)
    print(f'{split}: RoadFix={n_rf}, Signs={n_sg}, Lights={n_lt}, '
          f'Litter1={n_l1}, Litter2={n_l2}, '
          f'Bin1={n_b1}, Bin2={n_b2}, Bin3={n_b3}, Bin4={n_b4}')

# ── 6. Merge Mendeley faded markings ──────────────────────────
mendeley_map = {1: 6}
if os.path.exists(MENDELEY_DRIVE_PATH):
    all_pairs = []
    for subfolder in ['Attain_SMP_OS_V1.0', 'Attain_SMP_WS_V2.0']:
        img_dir = os.path.join(MENDELEY_DRIVE_PATH, subfolder, 'Images')
        lbl_dir = os.path.join(MENDELEY_DRIVE_PATH, subfolder, 'Labels')
        if not os.path.exists(img_dir):
            print(f'⚠️  Not found: {img_dir}')
            continue
        for img_path in glob.glob(os.path.join(img_dir, '*')):
            fname   = os.path.splitext(os.path.basename(img_path))[0]
            lbl_src = os.path.join(lbl_dir, fname + '.txt')
            if os.path.exists(lbl_src):
                all_pairs.append((img_path, lbl_src))
    print(f'Mendeley total images found: {len(all_pairs)}')
    random.seed(42)
    random.shuffle(all_pairs)
    n       = len(all_pairs)
    n_train = math.floor(n * 0.8)
    n_val   = math.floor(n * 0.1)
    splits  = (
        [(p, l, 'train') for p, l in all_pairs[:n_train]] +
        [(p, l, 'valid') for p, l in all_pairs[n_train:n_train+n_val]] +
        [(p, l, 'test')  for p, l in all_pairs[n_train+n_val:]]
    )
    for img_path, lbl_src, split in splits:
        fname = os.path.splitext(os.path.basename(img_path))[0]
        new_lines = []
        with open(lbl_src) as lf:
            for line in lf:
                parts = line.strip().split()
                if not parts: continue
                old_id = int(parts[0])
                new_id = mendeley_map.get(old_id)
                if new_id is None: continue
                coords = list(map(float, parts[1:]))
                xs = coords[0::2]; ys = coords[1::2]
                cx = (min(xs) + max(xs)) / 2
                cy = (min(ys) + max(ys)) / 2
                w  = max(xs) - min(xs)
                h  = max(ys) - min(ys)
                new_lines.append(f'{new_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
        if not new_lines: continue
        shutil.copy(img_path, os.path.join(MERGED_DIR, split, 'images'))
        with open(os.path.join(MERGED_DIR, split, 'labels', fname + '.txt'), 'w') as lf:
            lf.write('\n'.join(new_lines))
    print('Mendeley merged: 80/10/10 split across train/valid/test')
else:
    print('⚠️  Mendeley path not found:', MENDELEY_DRIVE_PATH)

# ── 7. Write merged data.yaml ──────────────────────────────────
merged_yaml = {
    'path':  MERGED_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(GLOBAL_CLASSES),
    'names': GLOBAL_CLASSES,
}
MERGED_YAML = os.path.join(MERGED_DIR, 'data.yaml')
with open(MERGED_YAML, 'w') as f:
    yaml.dump(merged_yaml, f, default_flow_style=False)
print('data.yaml written to:', MERGED_YAML)

# ── 8. Count images per split ──────────────────────────────────
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(os.path.join(MERGED_DIR, split, 'images', '*')))
    print(f'  {split}: {n} images')

DATA_YAML = MERGED_YAML
print('Ready — DATA_YAML =', DATA_YAML)

loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
RoadFix classes:  ['Fallen-Barrier', 'Fallen-Cone', 'Fallen-Pole', 'Pothole']
Signs classes:    ['deformation', 'dirty', 'graffiti', 'knocked', 'occluded', 'ok', 'perforation', 'stickers', 'worn']
Litter1 classes:  ['Aluminium foil', 'Bottle', 'Bottle cap', 'Can', 'Carton', 'Cigarette', 'Cup', 'Garbage bag', 'Glass jar', 'Lid', 'Other plastic', 'Paper', 'Paper bag', 'Plastic bag - wrapper', 'Plastic container', 'Plastic film', 'Plastic food container', 'Plastic utensils', '

## 🚀 Step 6 — Train Model (from v1 best.pt)

Starts training from your existing v1 `best.pt` weights (transfer learning — faster convergence than starting from COCO). The model head is automatically rebuilt for 7 classes.

> ⚠️ **Skip this cell** if you are resuming after a crash — use Step 7 or Step 8 instead.

In [ ]:
from ultralytics import YOLO
import os

# Use v1 best.pt as starting weights if available, else fall back to COCO pretrained
if USE_V1_WEIGHTS and os.path.exists(V1_BEST_PT):
    start_weights = V1_BEST_PT
    print(f'Starting from v1 best.pt: {start_weights}')
else:
    start_weights = 'yolo11n.pt'
    print('v1 weights not found — starting from COCO pretrained yolo11s.pt')

model = YOLO(start_weights)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    save_period=SAVE_EVERY,
    project=DRIVE_PROJECT_PATH,
    name=RUN_NAME,
    exist_ok=True,
    patience=PATIENCE,
    verbose=True,
)

BEST_PT_V2 = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt'
print(f'Training complete! Best model: {BEST_PT_V2}')


Starting from v1 best.pt: /content/drive/MyDrive/RoadSense/runs/roadsense_nano/weights/best.pt


CONVERT TO coreml (MLPACKAGE) FORMAT FOR IOS

In [ ]:
from ultralytics import YOLO

# Path to your best.pt (update if different)
model = YOLO('/content/drive/MyDrive/RoadSense/runs/roadfix_yolo11s/weights/best.pt')

# Export to Core ML with NMS (non‑maximum suppression) included
model.export(format='coreml', nms=True)

In [ ]:
import zipfile
import os
from google.colab import files

# Path to your .mlpackage folder
folder_path = '/content/drive/MyDrive/RoadSense/runs/roadfix_yolo11s/weights/best.mlpackage'

# Create a zip file in the current working directory
zip_path = '/content/best.mlpackage.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_folder in os.walk(folder_path):
        for file in files_in_folder:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, start=os.path.dirname(folder_path))
            zipf.write(full_path, arcname)

# Download the zip file to your Windows PC
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📊 Step 6b — Check Class Distribution

Before training, verify that each class has enough images. If any class has fewer than ~300 training images, that class will likely underperform. Consider finding more data or removing the class.

In [ ]:
import glob, os

print('Class distribution in merged training set:\n')
label_files = glob.glob(os.path.join(MERGED_DIR, 'train', 'labels', '*.txt'))
class_counts = {i: 0 for i in range(len(GLOBAL_CLASSES))}
for lf in label_files:
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

for idx, name in enumerate(GLOBAL_CLASSES):
    count = class_counts[idx]
    bar   = '#' * (count // 200)
    warn  = '⚠️  LOW' if count < 300 else ''
    print(f'  [{idx}] {name:<30} {count:>5} annotations  {bar} {warn}')


## 🔄 Step 7 — Resume After Crash (from last checkpoint)

Run this if Colab disconnected mid-training. Automatically continues from the last saved checkpoint.

**Before running this cell, re-run Steps 1 → 5 first** (install, mount drive, config, download dataset).

In [ ]:
from ultralytics import YOLO
import os

LAST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/last.pt'

if not os.path.exists(LAST_PT):
    print('❌ No checkpoint found at:', LAST_PT)
    print('   Run Step 6 (fresh training) instead.')
else:
    print(f'✅ Resuming from: {LAST_PT}')
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
    print('✅ Training resumed and completed.')


✅ Resuming from: /content/drive/MyDrive/RoadSense/runs/roadsense_v2_7class/weights/last.pt
Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/roadsense_merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/RoadSense/runs/roadsense_v2_7class/weights/l

KeyboardInterrupt: 

## 🎯 Step 8 — Resume From a Specific Checkpoint Epoch

Use this if you want to restart from a specific saved epoch (e.g. epoch 50) rather than the very last one.

Change `RESUME_FROM_EPOCH` to the epoch number you want to load.

In [ ]:
from ultralytics import YOLO
import os

RESUME_FROM_EPOCH = 50   # ← change to whichever epoch you want

checkpoint = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/epoch{RESUME_FROM_EPOCH}.pt'
weights_dir = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/'

if not os.path.exists(checkpoint):
    print(f'❌ epoch{RESUME_FROM_EPOCH}.pt not found.')
    if os.path.exists(weights_dir):
        available = sorted([f for f in os.listdir(weights_dir) if f.startswith('epoch')])
        print(f'   Available checkpoints: {available}')
    else:
        print('   No checkpoints found. Run Step 6 first.')
else:
    print(f'✅ Loading checkpoint: epoch{RESUME_FROM_EPOCH}.pt')
    model = YOLO(checkpoint)
    results = model.train(resume=True)
    print(f'✅ Resumed from epoch {RESUME_FROM_EPOCH} and completed.')


## 📊 Step 9 — Validate Model

Evaluates the best saved model on the validation set and prints mAP, precision, and recall.

In [ ]:
from ultralytics import YOLO

BEST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_PT)

metrics = model.val(data=DATA_YAML)

print(f'\n📊 Validation Results:')
print(f'   mAP50:     {metrics.box.map50:.3f}')
print(f'   mAP50-95:  {metrics.box.map:.3f}')
print(f'   Precision: {metrics.box.mp:.3f}')
print(f'   Recall:    {metrics.box.mr:.3f}')


Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,415,509 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1814.2±149.0 MB/s, size: 82.6 KB)
val: Scanning /content/roadsense_merged/valid/labels... 2807 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2807/2807 2.1Kit/s 1.4s
val: New cache created: /content/roadsense_merged/valid/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 438, len(boxes) = 4049. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 176/176 3.9it/s 45.2s
                   all       2807       4049      0.781      0.763      0.813      0.573
               Pothole        923       1776      0

## 🖼️ Step 10 — Test on a Sample Image

Runs inference on a test image and displays the result with bounding boxes drawn.

By default uses the first image from the test set. Change `TEST_IMAGE` to any path or URL.

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import glob, os

BEST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt'
CONF_THRESHOLD = 0.4   # ← lower = more detections, higher = more confident only

# Auto-pick first test image from dataset
test_imgs = glob.glob('/content/roadfix-3/test/images/*.jpg')
TEST_IMAGE = test_imgs[0] if test_imgs else 'https://ultralytics.com/images/bus.jpg'
print(f'Testing on: {TEST_IMAGE}')

model = YOLO(BEST_PT)
results = model.predict(source=TEST_IMAGE, conf=CONF_THRESHOLD, save=True,
                        project='/content', name='test_output', exist_ok=True)

saved = glob.glob('/content/test_output/**/*.jpg', recursive=True)
if saved:
    display(IPImage(saved[0], width=700))
else:
    print('No output image found. Check the prediction ran correctly.')


NameError: name 'DRIVE_PROJECT_PATH' is not defined